# 02 RAG Vector Search


## 경로


In [10]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data" / "sample_catalog.csv").exists():
  project_root = project_root.parent

data_path = project_root / "data" / "sample_catalog.csv"
env_path = project_root / ".env"

## 환경변수


In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(env_path)

if not os.getenv("OPENAI_API_KEY"):
  raise RuntimeError(".env 파일에 OPENAI_API_KEY를 입력해야 한다.")

chat_model = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.5")
embedding_model = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
embedding_dimensions = int(os.getenv("OPENAI_EMBEDDING_DIMENSIONS", "1536"))
top_k = int(os.getenv("RAG_TOP_K", "5"))

chat_model, embedding_model, embedding_dimensions, top_k


('gpt-5-nano', 'text-embedding-3-small', 1536, 5)

## 데이터


In [3]:
import pandas as pd

catalog = pd.read_csv(data_path)
catalog

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,1001,Men Blue Regular Fit Formal Shirt,Next Look,Men,1499,4,Regular fit solid blue formal shirt for office...,Blue
1,1002,Men White Regular Fit Formal Shirt,Van Heusen,Men,2199,5,White cotton formal shirt with regular fit and...,White
2,1003,Men Navy Slim Fit Formal Shirt,Peter England,Men,1799,5,Navy slim fit formal shirt for evening events,Navy
3,1004,Men Blue Printed Casual Shirt,ColorPlus,Men,1899,4,Blue and off-white printed casual shirt with r...,Blue
4,1005,Women Blue Regular Fit Shirt,Allen Solly,Women,1699,5,Blue regular fit shirt for smart casual office...,Blue
5,1006,Men White Linen Regular Fit Shirt,Raymond,Men,2499,3,White linen regular fit shirt suitable for for...,White
6,1007,Men Black Regular Fit Party Shirt,Blackberrys,Men,2299,4,Black regular fit shirt designed for party and...,Black
7,1008,Men Blue Regular Fit Oxford Shirt,Louis Philippe,Men,2799,6,Blue oxford shirt with regular fit for formal ...,Blue


## CSVLoader


In [4]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path=str(data_path), encoding="utf-8")
documents = loader.load()

print(f"documents: {len(documents)}")
print(documents[0].page_content)

documents: 8
ProductID: 1001
ProductName: Men Blue Regular Fit Formal Shirt
ProductBrand: Next Look
Gender: Men
Price (INR): 1499
NumImages: 4
Description: Regular fit solid blue formal shirt for office meetings and interviews
PrimaryColor: Blue


## Embedding


In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
  model=embedding_model,
  dimensions=embedding_dimensions,
)

sample_vector = embeddings.embed_query("Apple")
print(f"embedding dimensions: {len(sample_vector)}")
print(f"embedding preview: {sample_vector[:10]}")
print("embedding vector:")
print(sample_vector)


embedding dimensions: 1536


## FAISS index


In [6]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents, embeddings)
vector_store.index.ntotal


8

## Similarity search


In [ ]:
query = (
  "Shirts which are good for Men, have regular fit and not the slim fit, "
  "can be used for a formal occasion, and have a color of either Blue or White"
)

results = vector_store.similarity_search_with_score(query, k=top_k)

for rank, (doc, score) in enumerate(results, start=1):
  print(f"rank={rank} score={score:.4f}")
  print(doc.page_content)
  print("-" * 80)


rank=1 score=0.8924
ProductID: 1001
ProductName: Men Blue Regular Fit Formal Shirt
ProductBrand: Next Look
Gender: Men
Price (INR): 1499
NumImages: 4
Description: Regular fit solid blue formal shirt for office meetings and interviews
PrimaryColor: Blue
--------------------------------------------------------------------------------
rank=2 score=0.8936
ProductID: 1003
ProductName: Men Navy Slim Fit Formal Shirt
ProductBrand: Peter England
Gender: Men
Price (INR): 1799
NumImages: 5
Description: Navy slim fit formal shirt for evening events
PrimaryColor: Navy
--------------------------------------------------------------------------------
rank=3 score=0.8950
ProductID: 1002
ProductName: Men White Regular Fit Formal Shirt
ProductBrand: Van Heusen
Gender: Men
Price (INR): 2199
NumImages: 5
Description: White cotton formal shirt with regular fit and spread collar
PrimaryColor: White
--------------------------------------------------------------------------------
rank=4 score=0.9498
ProductID

## Cosine similarity


In [8]:
import numpy as np

def cosine_similarity(left: list[float], right: list[float]) -> float:
  left_vector = np.array(left)
  right_vector = np.array(right)
  return float(np.dot(left_vector, right_vector) / (np.linalg.norm(left_vector) * np.linalg.norm(right_vector)))

query_vector = embeddings.embed_query(query)
document_vectors = embeddings.embed_documents([doc.page_content for doc in documents])

manual_scores = [
  (idx, cosine_similarity(query_vector, document_vector), documents[idx])
  for idx, document_vector in enumerate(document_vectors)
]

for idx, score, doc in sorted(manual_scores, key=lambda item: item[1], reverse=True)[:top_k]:
  print(f"row={idx} cosine={score:.4f}")
  print(doc.page_content)
  print("-" * 80)


row=0 cosine=0.5537
ProductID: 1001
ProductName: Men Blue Regular Fit Formal Shirt
ProductBrand: Next Look
Gender: Men
Price (INR): 1499
NumImages: 4
Description: Regular fit solid blue formal shirt for office meetings and interviews
PrimaryColor: Blue
--------------------------------------------------------------------------------
row=2 cosine=0.5532
ProductID: 1003
ProductName: Men Navy Slim Fit Formal Shirt
ProductBrand: Peter England
Gender: Men
Price (INR): 1799
NumImages: 5
Description: Navy slim fit formal shirt for evening events
PrimaryColor: Navy
--------------------------------------------------------------------------------
row=1 cosine=0.5524
ProductID: 1002
ProductName: Men White Regular Fit Formal Shirt
ProductBrand: Van Heusen
Gender: Men
Price (INR): 2199
NumImages: 5
Description: White cotton formal shirt with regular fit and spread collar
PrimaryColor: White
--------------------------------------------------------------------------------
row=5 cosine=0.5252
ProductID

## Context prompt


In [13]:
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=chat_model)

context = "".join(
  f"[Document {rank}]\n{doc.page_content}\n"
  for rank, (doc, _score) in enumerate(results, start=1)
)

prompt = f"""
You are a product recommendation assistant.
Use only the product context below. If the context is insufficient, say so.

Product context:
{context}

Question:
{query}

Respond in Korean. First provide a compact markdown table, then recommend one product with a reason.
""".strip()

response = llm.invoke(prompt)
display(Markdown(response.content))


| ProductID | ProductName | Brand | Color | Fit | Price (INR) |
|-----------|-------------|-------|-------|-----|-------------|
| 1001 | Men Blue Regular Fit Formal Shirt | Next Look | Blue | Regular Fit | 1499 |
| 1002 | Men White Regular Fit Formal Shirt | Van Heusen | White | Regular Fit | 2199 |
| 1006 | Men White Linen Regular Fit Shirt | Raymond | White | Regular Fit | 2499 |
| 1008 | Men Blue Regular Fit Oxford Shirt | Louis Philippe | Blue | Regular Fit | 2799 |

추천 상품: 1002번 Men White Regular Fit Formal Shirt (Van Heusen)

이유: 화이트 색상은 포멀한 자리에 가장 표준적이고 다용도로 활용도가 높습니다. 정규 핏으로 편안한 착용감을 주며, 다양한 수트나 정장 아이템과 무난하게 조화됩니다. 브랜드도 신뢰도가 높고, 가격 또한 합리적인 편이라 기본 포멀 셔츠로 추천하기 적합합니다.

## 개인화 조건


In [ ]:
demographics = {
  "John": {
    "Age": 30,
    "Education": "Bachelor's Degree in Computer Science",
    "Occupation": "Software Engineer",
  },
  "Adam": {
    "Age": 60,
    "Occupation": "Retired",
  },
}

personalized_prompt = f"""
Use only the product context below.

Product context:
{context}

Customer profiles:
{demographics}

Question:
{query}

Recommend one shirt for each customer. Respond in Korean and explain why each recommendation matches the customer.
""".strip()

personalized_response = llm.invoke(personalized_prompt)
display(Markdown(personalized_response.content))


## 확인 질문

- Top-K 문서 중 답변에 쓰인 문서는 무엇인가?
- Top-K를 2로 줄이면 답변이 어떻게 바뀌는가?
- `OPENAI_CHAT_MODEL`을 바꾸면 비용, 속도, 품질이 어떻게 바뀌는가?
